In [ ]:
import numpy as np
import pandas as pd
import os
import time
import torch
import glob
import nltk
import transformers
import pytorch_lightning as pl
import torch.nn.functional as F
# from sklearn.metrics import rouge_score
from rouge_score import rouge_scorer
from nltk import tokenize
from pytorch_lightning.callbacks import ModelCheckpoint
from torch.utils.data import DataLoader, TensorDataset, RandomSampler
from transformers import BartTokenizer, BartForConditionalGeneration

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

class LitModel(pl.LightningModule):
    def __init__(self, learning_rate, tokenizer, model):
        super().__init__()
        self.tokenizer = tokenizer
        self.model = model.to(device)
        self.learning_rate = learning_rate

        self.hparams.freeze_encoder = True
        self.hparams.freeze_embeds = True
        self.hparams.eval_beams = 4

        if self.hparams.freeze_encoder:
            freeze_params(self.model.get_encoder())

        if self.hparams.freeze_embeds:
            self.freeze_embeds()

    def freeze_embeds(self):
        '''Freeze the positional embedding parameters of the model'''
        freeze_params(self.model.model.shared)
        for d in [self.model.model.encoder, self.model.model.decoder]:
            freeze_params(d.embed_positions)
            freeze_params(d.embed_tokens)

    def forward(self, input_ids, **kwargs):
        return self.model(input_ids, **kwargs)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        return optimizer

    def training_step(self, batch, batch_idx):
        src_ids, src_mask, tgt_ids = batch
        decoder_input_ids = shift_tokens_right(tgt_ids, self.tokenizer.pad_token_id)
        outputs = self(src_ids, attention_mask=src_mask, decoder_input_ids=decoder_input_ids, use_cache=False)
        lm_logits = outputs[0]
        ce_loss_fct = torch.nn.CrossEntropyLoss(ignore_index=self.tokenizer.pad_token_id)
        loss = ce_loss_fct(lm_logits.view(-1, lm_logits.shape[-1]), tgt_ids.view(-1))
        return {'loss': loss}

    def validation_step(self, batch, batch_idx):
        src_ids, src_mask, tgt_ids = batch
        decoder_input_ids = shift_tokens_right(tgt_ids, self.tokenizer.pad_token_id)
        outputs = self(src_ids, attention_mask=src_mask, decoder_input_ids=decoder_input_ids, use_cache=False)
        lm_logits = outputs[0]
        ce_loss_fct = torch.nn.CrossEntropyLoss(ignore_index=self.tokenizer.pad_token_id)
        val_loss = ce_loss_fct(lm_logits.view(-1, lm_logits.shape[-1]), tgt_ids.view(-1))
        return {'loss': val_loss}

    def generate_text(self, text, eval_beams, early_stopping=True, max_len=1024):
        generated_ids = self.model.generate(
            text["input_ids"],
            attention_mask=text["attention_mask"],
            use_cache=True,
            decoder_start_token_id=self.tokenizer.pad_token_id,
            num_beams=eval_beams,
            max_length=max_len,
            early_stopping=early_stopping
        )
        return [self.tokenizer.decode(w, skip_special_tokens=True, clean_up_tokenization_spaces=True) for w in generated_ids]

In [ ]:
def freeze_params(model):
    for layer in model.parameters():
        layer.requires_grad = False

class SummaryDataModule(pl.LightningDataModule):
    def __init__(self, tokenizer, df, batch_size):
        super().__init__()
        self.tokenizer = tokenizer
        self.batch_size = batch_size
        self.data = df

    def prepare_data(self):
        self.train, self.validate, self.test = np.split(self.data.sample(frac=1), [int(.6 * len(self.data)), int(.8 * len(self.data))])

    def setup(self, stage):
        self.train = encode_sentences(self.tokenizer, self.train['source'], self.train['target'])
        self.validate = encode_sentences(self.tokenizer, self.validate['source'], self.validate['target'])
        self.test = encode_sentences(self.tokenizer, self.test['source'], self.test['target'])

    def train_dataloader(self):
        dataset = TensorDataset(self.train['input_ids'], self.train['attention_mask'], self.train['labels'])
        return DataLoader(dataset, sampler=RandomSampler(dataset), batch_size=self.batch_size)

    def val_dataloader(self):
        dataset = TensorDataset(self.validate['input_ids'], self.validate['attention_mask'], self.validate['labels'])
        return DataLoader(dataset, batch_size=self.batch_size)

    def test_dataloader(self):
        dataset = TensorDataset(self.test['input_ids'], self.test['attention_mask'], self.test['labels'])
        return DataLoader(dataset, batch_size=self.batch_size)

In [ ]:
def shift_tokens_right(input_ids, pad_token_id):
    prev_output_tokens = input_ids.clone()
    index_of_eos = (input_ids.ne(pad_token_id).sum(dim=1) - 1).unsqueeze(-1)
    prev_output_tokens[:, 0] = input_ids.gather(1, index_of_eos).squeeze()
    prev_output_tokens[:, 1:] = input_ids[:, :-1]
    return prev_output_tokens

def encode_sentences(tokenizer, source_sentences, target_sentences, max_length=1024, min_length=1024, pad_to_max_length=True, return_tensors="pt"):
    input_ids, attention_masks, target_ids = [], [], []

    for sentence in source_sentences:
        encoded_dict = tokenizer(sentence, max_length=max_length, padding="max_length" if pad_to_max_length else None,
                                 truncation=True, return_tensors=return_tensors, add_prefix_space=True)
        input_ids.append(encoded_dict['input_ids'])
        attention_masks.append(encoded_dict['attention_mask'])

    for sentence in target_sentences:
        encoded_dict = tokenizer(sentence, max_length=min_length, padding="max_length" if pad_to_max_length else None,
                                 truncation=True, return_tensors=return_tensors, add_prefix_space=True)
        target_ids.append(encoded_dict['input_ids'])

    return {
        "input_ids": torch.cat(input_ids, dim=0),
        "attention_mask": torch.cat(attention_masks, dim=0),
        "labels": torch.cat(target_ids, dim=0)
    }

In [ ]:
# Define the directory where your files are located
directory = 'dataset/IN-Abs/train-data/judgement'

# Get all file names in the directory and filter out directories
names = [name for name in os.listdir(directory) if os.path.isfile(os.path.join(directory, name))]

# Load the content of the files into data_source using 'latin-1' encoding
data_source = []
for name in names:
    with open(os.path.join(directory, name), 'r', encoding='latin-1') as file:
        data_source.append(file.read())

# Now proceed with your summarization loop
for i, (name, doc) in enumerate(zip(names, data_source)):
    wc = doc.split(" ")  # Split the document into words
    input_len = len(wc)  # Get the word count
    print(f"Processing document {i+1}/{len(names)}: {name} with {input_len} words")

    # Your summarization logic here
    # Example: Assuming summarization logic or model goes here
    # summary = summarize(doc)
    # print(summary)  # Or save it to a file if needed


# Assuming you have the device set up for GPU usage
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Define dataset and output path
dataset = "IN"
output_path = "./outputs/output-main/"
if not os.path.exists(output_path):
    os.makedirs(output_path)

In [ ]:
# Load model and tokenizer
tokenizer = BartTokenizer.from_pretrained('facebook/bart-large', add_prefix_space=True)
model = BartForConditionalGeneration.from_pretrained("facebook/bart-large")

# Fine-tuned model loading
class LitModel:
    def __init__(self, learning_rate, tokenizer, model):
        self.learning_rate = learning_rate
        self.tokenizer = tokenizer
        self.model = model

bart_model = LitModel(learning_rate=2e-5, tokenizer=tokenizer, model=model)

# Function to generate summaries on GPU
def generate_summary_gpu(nested_sentences, p=0.2):
    summaries = []
    for nested in nested_sentences:
        l = int(p * len(nested.split(" ")))  # Calculate the desired length
        input_tokenized = tokenizer.encode(nested, truncation=True, return_tensors='pt').to(device)
        summary_ids = bart_model.model.to(device).generate(
            input_tokenized, length_penalty=0.01, min_length=l-5, max_length=l+5
        )
        output = [tokenizer.decode(g, skip_special_tokens=True, clean_up_tokenization_spaces=False) for g in summary_ids]
        summaries.append(output)
    return [sentence for sublist in summaries for sentence in sublist]

def calculate_rouge(reference, hypothesis):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(reference, hypothesis)
    return scores

In [ ]:
# Path to training data
directory = './inputs/input-main/'

# Get all file names in the directory
names = os.listdir(directory)

# Filter out directories from the list of names (if any)
names = [name for name in names if os.path.isfile(os.path.join(directory, name))]

# Load the content of the files using 'latin-1' encoding to avoid encoding issues
data_source = []
for name in names:
    with open(os.path.join(directory, name), 'r', encoding='latin-1') as file:
        data_source.append(file.read())

# If you have specific required lengths for each document, define dict_names (optional)
dict_names = {
    # 'filename.txt': required_summary_length,  # Example
    # Add entries for each file if needed
}

# Main loop for generating summaries
# for i, (name, doc) in enumerate(zip(names, data_source)):
#     wc = doc.split(" ")  # Split document by words
#     input_len = len(wc)  # Total word count of the document
#     req_len = dict_names.get(name, get_required_summary_length(input_len))  # Get required summary length

#     print(f"{i}: {name} - {input_len} words, required summary length: {req_len} words")

#     # Nest the document into smaller chunks
#     nested = nest_sentences(doc, 1024)

#     # Calculate the proportion of words to retain for the summary
#     p = float(req_len / input_len)

#     # Generate the abstracted summary
#     abs_summ = generate_summary_gpu(nested, p)
#     abs_summ = " ".join(abs_summ[:req_len])  # Truncate summary to required length

#     # Write the summary to the output directory
#     with open(os.path.join(output_path, name), 'w') as file:py

# Initialize performance metrics
total_time = 0
total_rouge1 = 0
total_rouge2 = 0
total_rougeL = 0
total_documents = len(names)

# Main loop for generating summaries
for i, (name, doc) in enumerate(zip(names, data_source)):
    try:
        wc = doc.split(" ")  # Split document by words
        input_len = len(wc)  # Total word count of the document
        req_len = dict_names.get(name, get_required_summary_length(input_len))  # Get required summary length

        print(f"{i}: {name} - {input_len} words, required summary length: {req_len} words")

        # Nest the document into smaller chunks
        nested = nest_sentences(doc, 1024)

        # Calculate the proportion of words to retain for the summary
        p = float(req_len / input_len)

        # Generate the abstracted summary
        start_time = time.time()
        abs_summ = generate_summary_gpu(nested, p)
        abs_summ = " ".join(abs_summ[:req_len])  # Truncate summary to required length
        end_time = time.time()

        # Calculate ROUGE scores
        rouge_scores = calculate_rouge(doc, abs_summ)

        # Update performance metrics
        total_time += end_time - start_time
        total_rouge1 += rouge_scores['rouge1'].fmeasure
        total_rouge2 += rouge_scores['rouge2'].fmeasure
        total_rougeL += rouge_scores['rougeL'].fmeasure

        # Write the summary to the output directory
        with open(os.path.join(output_path, name), 'w') as file:
            file.write(abs_summ)

    except Exception as e:
        print(f"Error processing document {name}: {str(e)}")
        continue

In [ ]:
##WORKS CORRECT $#1
# # Main loop for generating summaries
# for i, (name, doc) in enumerate(zip(names, data_source)):
#     wc = doc.split(" ")  # Split document by words
#     input_len = len(wc)  # Total word count of the document
#     req_len = dict_names.get(name, get_required_summary_length(input_len))  # Get required summary length

#     print(f"{i}: {name} - {input_len} words, required summary length: {req_len} words")

#     # Nest the document into smaller chunks
#     nested = nest_sentences(doc, 1024)

#     # Calculate the proportion of words to retain for the summary
#     p = float(req_len / input_len)

#     # Generate the abstracted summary
#     start_time = time.time()
#     abs_summ = generate_summary_gpu(nested, p)
#     abs_summ = " ".join(abs_summ[:req_len])  # Truncate summary to required length
#     end_time = time.time()

#     # Calculate ROUGE scores
#     rouge_scores = calculate_rouge(doc, abs_summ)

#     # Update performance metrics
#     total_time += end_time - start_time
#     total_rouge1 += rouge_scores['rouge1'].fmeasure
#     total_rouge2 += rouge_scores['rouge2'].fmeasure
#     total_rougeL += rouge_scores['rougeL'].fmeasure

#     # Write the summary to the output directory
#     with open(os.path.join(output_path, name), 'w') as file:
#         file.write(abs_summ)

# Calculate average performance metrics
avg_time = total_time / total_documents
avg_rouge1 = total_rouge1 / total_documents
avg_rouge2 = total_rouge2 / total_documents
avg_rougeL = total_rougeL / total_documents

# Display performance matrix
print("\nPerformance Matrix:")
print(f"Average processing time per document: {avg_time:.2f} seconds")
print(f"Average ROUGE-1 F1-score: {avg_rouge1:.4f}")
print(f"Average ROUGE-2 F1-score: {avg_rouge2:.4f}")
print(f"Average ROUGE-L F1-score: {avg_rougeL:.4f}")